<a href="https://colab.research.google.com/github/mudassar2224/Multiplicative-Attention-Luong------LLM----Journey/blob/main/Multiplicative_Attention_(Luong).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# $\color{crimson}{\text{**Import Libraries** }}$

In [19]:
import numpy as np ,pandas as pd
import string
from string import  punctuation, digits
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input , LSTM,Dense,Concatenate,Embedding,  AdditiveAttention
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive

## $\color{crimson}{\text{Load Dataset }}$

In [20]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
lines=pd.read_csv("/content/drive/MyDrive/Hindi_English_Truncated_Corpus.csv")
lines

,source,english_sentence,hindi_sentence
0,ted,politicians do not have permission to do what ...,"राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह कर..."
1,ted,"I'd like to tell you about one such child,",मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहू...
2,indic2012,This percentage is even greater than the perce...,यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।
3,ted,what we really mean is that they're bad at not...,हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते
4,indic2012,.The ending portion of these Vedas is called U...,इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।
...,...,...,...
127602,indic2012,Examples of art deco construction can be found...,आर्ट डेको शैली के निर्माण मैरीन ड्राइव और ओवल ...
127603,ted,and put it in our cheeks.,और अपने गालों में डाल लेते हैं।
127604,tides,"As for the other derivatives of sulphur , the ...","जहां तक गंधक के अन्य उत्पादों का प्रश्न है , द..."
127605,tides,its complicated functioning is defined thus in...,Zरचना-प्रकिया को उसने एक पहेली में यों बांधा है .


In [22]:
# Filter use Only source
if "source" in lines.columns:
    lines=lines[lines["source"]=="ted"][["english_sentence", "hindi_sentence"]].dropna().drop_duplicates()
lines=lines.sample(n=min(len(lines), 2500), random_state=42)

In [23]:
lines

,english_sentence,hindi_sentence
82040,"We still don't know who her parents are, who s...",हम अभी तक नहीं जानते हैं कि उसके माता-पिता कौन...
85038,"no keyboard,","कोई कुंजीपटल नहीं,"
58018,"But as far as being a performer,",लेकिन एक कलाकार होने के साथ
74470,"And this particular balloon,","और यह खास गुब्बारा,"
122330,and it's not as hard as you think. Integrate c...,"और जितना आपको लगता है, यह उतना कठिन नहीं है.अप..."
...,...,...
113839,to try putting it out onto the other location.,कि उसे दूसरी जगह रखे।
83509,So this is just one example,तो यह सिर्फ एक उदाहरण है
112988,"sitting in their homes,",मुझे इन्टरनेट पर
33974,I carried my payload back downstairs,मैं अपना सामान लेकर नीचे की तरफ गया



## $\color{crimson}{\text{Text Cleanig }}$

In [24]:
def claen_text(text):
  text=str(text).lower()
  for char in string.punctuation:
    text=text.replace(char ,"")
  for digit in string.digits:
    text=text.replace( digit, "")
  return text.strip()
lines["english_sentence"]=lines["english_sentence"].apply(claen_text)
lines["hindi_sentence"]=lines["hindi_sentence"].apply(claen_text)

In [25]:
print(lines["english_sentence"])
print(lines["hindi_sentence"])

82040     we still dont know who her parents are who she is
85038                                           no keyboard
58018                       but as far as being a performer
74470                           and this particular balloon
122330    and its not as hard as you think integrate cli...
                                ...                        
113839        to try putting it out onto the other location
83509                           so this is just one example
112988                               sitting in their homes
33974                  i carried my payload back downstairs
119369    has inspired human beings to think beyond the ...
Name: english_sentence, Length: 2500, dtype: object
82040     हम अभी तक नहीं जानते हैं कि उसके मातापिता कौन ...
85038                                     कोई कुंजीपटल नहीं
58018                           लेकिन एक कलाकार होने के साथ
74470                                    और यह खास गुब्बारा
122330    और जितना आपको लगता है यह उतना कठिन नही

## $\color{crimson}{\text{Tokenizer and Vocabulry }}$

In [26]:
eng_tokenizer=Tokenizer()
eng_tokenizer.fit_on_texts(lines["english_sentence"])
eng_seq=eng_tokenizer.texts_to_sequences(lines["english_sentence"])
# For Hindi
hind_tokenizer=Tokenizer()
hind_tokenizer.fit_on_texts(lines["hindi_sentence"])
hind_seq=hind_tokenizer.texts_to_sequences(lines["hindi_sentence"])
# Size of Vacbulary
eng_vocab_size=len(eng_tokenizer.word_index)+1
print(f"English Vocabulay Size : { eng_vocab_size}")
hin_vocab_size=len(hind_tokenizer.word_index)+1
print(f"Hindi Vocabulary Size : {hin_vocab_size} ")

English Vocabulay Size : 3843
Hindi Vocabulary Size : 4556 


## $\color{crimson}{\text{Add Padding  }}$

In [27]:
max_len_eng=max(len(seq) for seq in eng_seq)
print(max_len_eng)
max_len_hind=max(len(seq) for seq in hind_seq )
print(max_len_hind)

20
27


In [28]:
encoder_input=pad_sequences(eng_seq, maxlen=max_len_eng, padding="post")
decoder_input=pad_sequences(hind_seq, maxlen=max_len_hind, padding="post")
print(encoder_input.shape)
print(decoder_input.shape)

(2500, 20)
(2500, 27)


In [29]:
# Create A 3 D Array For Sparse Categorical Cross Entropy
decoder_target=np.zeros((decoder_input.shape[0], decoder_input.shape[1], 1))
decoder_target[: , 0:-1 ,0 ]=decoder_input[: , 1:]

# $\color{crimson}{\text**{**Multiplicative Attention (Luong) in
odel**  }**}$

In [30]:
latent_dim=512
# Encoder Architecture
encoder_input=Input(shape=(None, ), name="encoder_input")

# Define the Embedding layer object
encoder_embedding_layer = Embedding(eng_vocab_size, latent_dim, name="encoder_embedding")
# Apply the Embedding layer to the encoder_input tensor
encoder_embedded_output = encoder_embedding_layer(encoder_input)

# LSTM Design
encoder_lstm=LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    dropout=0.2
    ,
    recurrent_dropout=0.2  ,

    name="encoder_lstm"

)

# Pass the output of the embedding layer to the LSTM layer
encoder_outpts, state_h, state_c = encoder_lstm(encoder_embedded_output)
encoder_states=[state_h, state_c]

In [31]:
# Decoder
decoder_input=Input(shape=(None, ), name="decoder_input")
decoder_embed_layer=Embedding(hin_vocab_size, latent_dim,name="deccoder_embeddings")(decoder_input)
decoder_lstm=LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    dropout=0.2
    ,

    name="decoder_lstm"
)
decoder_outputs , _ , _ = decoder_lstm(decoder_embed_layer, initial_state=encoder_states)



In [32]:
# Multiplicative Attention (Luong) in Apply here

from tensorflow.keras.layers import Attention

# Luong / Multiplicative (Dot-Product) Attention
attention_layer = Attention(use_scale=True, name="luong_attention")
context_vector = attention_layer([decoder_outputs, encoder_outpts])
decoder_combined_context=Concatenate(axis=-1,  name="concat_layer" )([decoder_outputs, context_vector])


In [33]:
# dense Layer ForHindi
decoder_dense=Dense(hin_vocab_size, activation="softmax", name="output_layer")
decoder_final_output=decoder_dense(decoder_combined_context)


In [34]:

#Combine into Model
model = Model([encoder_input, decoder_input], decoder_final_output)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Ensure X_encoder and X_decoder are defined for model.fit
X_encoder = pad_sequences(eng_seq, maxlen=max_len_eng, padding="post")
X_decoder = pad_sequences(hind_seq, maxlen=max_len_hind, padding="post")

history = model.fit(
    [X_encoder, X_decoder],
    decoder_target,
    batch_size=64,
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stop]
)


Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 221s 7s/step - accuracy: 0.6966 - loss: 2.8609 - val_accuracy: 0.7286 - val_loss: 2.0314
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 258s 6s/step - accuracy: 0.7212 - loss: 1.9647 - val_accuracy: 0.7291 - val_loss: 2.0153
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 211s 7s/step - accuracy: 0.7221 - loss: 1.9126 - val_accuracy: 0.7290 - val_loss: 2.0175
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 207s 6s/step - accuracy: 0.7230 - loss: 1.8702 - val_accuracy: 0.7294 - val_loss: 2.0142
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 205s 6s/step - accuracy: 0.7253 - loss: 1.8200 - val_accuracy: 0.7311 - val_loss: 2.0110
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 273s 7s/step - accuracy: 0.7289 - loss: 1.7601 - val_accuracy: 0.7342 - val_loss: 2.0119
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 212s 7s/step - accuracy: 0.7309 - loss: 1.6975 - val_accuracy: 0.7367 - val_loss: 2.0190
Epoch 8/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 272s 7s/step - accuracy: 0.7344 - loss: 1.6337 - val_accuracy: 0.7359 - v

## $\color{crimson}{\text{Definne Inferene }}$

In [35]:
# ----------------------------------------------------
# 1. ENCODER INFERENCE MODEL
# ----------------------------------------------------
encoder_model_inf = Model(
    inputs=encoder_input,
    outputs=[encoder_outpts, state_h, state_c]
)

# ----------------------------------------------------
# 2. DECODER INFERENCE MODEL
# ----------------------------------------------------
# Inputs for decoder hidden/cell states
decoder_state_input_h = Input(shape=(latent_dim,), name="inf_dec_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="inf_dec_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Input for encoder outputs (for computing attention)
encoder_outputs_input = Input(shape=(max_len_eng, latent_dim), name="inf_enc_outputs")

# Get the actual decoder embedding layer object from the trained model
decoder_embedding_layer_obj = model.get_layer('deccoder_embeddings')

# Pass target token through decoder embedding
dec_inf_emb = decoder_embedding_layer_obj(decoder_input)

# Pass through decoder LSTM
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_inf_emb,
    initial_state=decoder_states_inputs
)

# Compute Attention Context
context_vector_inf = attention_layer([decoder_outputs_inf, encoder_outputs_input])

# Concatenate Context & Decoder Output
decoder_combined_inf = Concatenate(axis=-1)([decoder_outputs_inf, context_vector_inf])

# Dense layer for next word probabilities
decoder_outputs_inf = decoder_dense(decoder_combined_inf)

# Combine into Decoder Inference Model
decoder_model_inf = Model(
    inputs=[decoder_input, encoder_outputs_input] + decoder_states_inputs,
    outputs=[decoder_outputs_inf, state_h_inf, state_c_inf]
)

In [36]:
reverse_eng = {idx: word for word, idx in eng_tokenizer.word_index.items()}
reverse_hin = {idx: word for word, idx in hind_tokenizer.word_index.items()}
reverse_hin[0] = ''  # Handle padding index

In [37]:
def decode_sequence(input_seq):
    # 1. Encode input sequence to get encoder outputs + initial hidden states
    enc_out, state_h_val, state_c_val = encoder_model_inf.predict(input_seq, verbose=0)
    states_value = [state_h_val, state_c_val]

    # 2. Start target sequence with start token 'start_'
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hind_tokenizer.word_index.get('start_', 1)

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        # Predict next token using decoder inference model
        output_tokens, h, c = decoder_model_inf.predict(
            [target_seq, enc_out] + states_value,
            verbose=0
        )

        # Select word with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_hin.get(sampled_token_index, '')

        # Stop condition: reached end token '_end' or maximum length limit
        if sampled_word == '_end' or len(decoded_sentence) >= max_hind_len:
            stop_condition = True
        elif sampled_word != '':
            decoded_sentence.append(sampled_word)

        # Update input token and hidden states for next prediction step
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return " ".join(decoded_sentence)

In [ ]:
max_hind_len = max_len_hind  # Fix the variable name reference

for i in range(5):
    sample_eng_seq = X_encoder[i:i+1]

    # Reconstruct original English sentence
    orig_eng = " ".join([reverse_eng.get(idx, '') for idx in sample_eng_seq[0] if idx > 0])

    # Predict Hindi translation
    translated_hindi = decode_sequence(sample_eng_seq)

    print(f"English:    {orig_eng}")
    print(f"Hindi Pred: {translated_hindi}")
    print("-" * 50)